# Análise Exploratória da Carga Elétrica

## 1. Contexto do Projeto

A carga elétrica representa a potência média demandada do sistema elétrico durante determinado período. Nesta base, cada valor corresponde à carga média registrada durante uma hora e está expresso em MWmed.

Neste projeto, serão analisados os registros horários de carga elétrica dos subsistemas Norte, Nordeste, Sul e Sudeste, disponibilizados pelo Operador Nacional do Sistema Elétrico (ONS).

O período analisado compreende os anos de 2000 a 2025.

## 2. Objetivo da Análise

O objetivo desta análise exploratória é compreender como a carga elétrica brasileira se comportou ao longo do período disponível.

A análise buscará responder às seguintes perguntas:

- Como a carga elétrica evoluiu ao longo dos anos?
- Quais subsistemas apresentam as maiores cargas?
- Os subsistemas possuem comportamentos diferentes?
- Em quais meses a carga costuma ser maior ou menor?
- Existe diferença entre dias úteis e finais de semana?
- Quais horários apresentam maior demanda?
- Quais foram os principais picos e as menores cargas registradas?
- A variabilidade da carga mudou ao longo dos anos?
- Existem períodos com comportamentos que merecem uma investigação mais detalhada?

Além da análise temporal, também serão utilizadas medidas estatísticas para compreender a distribuição, a posição e a dispersão dos dados.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = 'plotly_white'
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/processed')

electric_data = pd.read_parquet(DATA_PATH / 'curva_carga_consolidado.parquet')


In [3]:
electric_data.head()

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
0,N,NORTE,2000-01-01 00:00:00,2373.7
1,NE,NORDESTE,2000-01-01 00:00:00,5340.2
2,S,SUL,2000-01-01 00:00:00,5777.0
3,SE,SUDESTE,2000-01-01 00:00:00,21183.0
4,N,NORTE,2000-01-01 01:00:00,2331.6


In [4]:
electric_data.info(memory_usage='deep')


<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911348 non-null  float64       
dtypes: datetime64[ns](1), float64(1), str(2)
memory usage: 34.1 MB


In [5]:
print(f'O dataset possui um total de {electric_data.shape[0]} linhas e {electric_data.shape[1]} colunas.')

O dataset possui um total de 911608 linhas e 4 colunas.


In [6]:
print(f"Período analisado: {electric_data['din_instante'].min()} até {electric_data['din_instante'].max()}")


Período analisado: 2000-01-01 00:00:00 até 2025-12-31 23:00:00


In [7]:
electric_data[['id_subsistema', 'nom_subsistema']].drop_duplicates().sort_values('id_subsistema')

,id_subsistema,nom_subsistema
0,N,NORTE
1,NE,NORDESTE
2,S,SUL
3,SE,SUDESTE


O dataset consolidado contém registros horários dos quatro subsistemas durante o período de 2000 a 2025.

As verificações relacionadas a tipos de dados, valores ausentes, registros duplicados e continuidade temporal foram realizadas anteriormente no notebook de consolidação. Portanto, neste notebook, o foco será o comportamento da carga elétrica.

## 3. Panorama Estatístico da Carga Elétrica

Antes de analisar a evolução da carga ao longo do tempo, vamos observar algumas medidas estatísticas de cada subsistema.

Essa primeira análise permite comparar a magnitude e a dispersão dos registros. Como os dados abrangem 26 anos, os resultados representam um panorama geral. As mudanças ocorridas durante o período serão analisadas posteriormente.

In [8]:
view_describe = electric_data.groupby(['id_subsistema', 'nom_subsistema'])['val_cargaenergiahomwmed'].describe().round(2)
view_describe

,,count,mean,std,min,25%,50%,75%,max
id_subsistema,nom_subsistema,,,,,,,,
N,NORTE,227837.0,4592.68,1670.83,610.40,3275.08,4284.52,5640.59,10239.23
NE,NORDESTE,227837.0,8886.53,2478.98,149.86,6826.40,8768.52,10710.88,18156.97
S,SUL,227837.0,9737.16,2860.97,3615.83,7621.37,9392.76,11762.64,22737.44
SE,SUDESTE,227837.0,33826.25,7625.22,7970.58,28174.30,33508.33,39214.42,62149.88


A carga elétrica apresenta magnitudes bastante diferentes entre os subsistemas. O SUDESTE possui os maiores valores de média, mediana e quartis, enquanto o NORTE apresenta os menores. Entre eles, aparecem os subsistemas SUL e NORDESTE.

Em todos os subsistemas, a média é um pouco maior que a mediana. Isso indica que os valores mais altos podem estar elevando a média, mas ainda precisamos observar a distribuição dos dados antes de confirmar esse comportamento.

O desvio-padrão do SUDESTE também é o maior em termos absolutos. Entretanto, isso não significa necessariamente que ele seja o subsistema mais variável, pois sua carga média também é muito superior às demais. Para fazer uma comparação proporcional, será calculado o coeficiente de variação.

A coluna `count` apresenta 227.837 valores válidos em cada subsistema. Como o método `describe()` desconsidera valores nulos, essa igualdade representa apenas a quantidade de cargas válidas, não garante que os quatro subsistemas tenham exatamente os mesmos horários disponíveis.

### 3.1 Medidas Adicionais de Dispersão

Como os subsistemas possuem cargas de magnitudes diferentes, apenas o desvio-padrão não é suficiente para comparar a variabilidade entre eles.

Por isso, também serão calculadas:

- **Variância:** mede o quanto os valores se afastam da média;
- **Amplitude:** diferença entre a maior e a menor carga;
- **Intervalo interquartil:** distância entre o primeiro e o terceiro quartil, representando a dispersão dos 50% centrais dos dados;
- **Coeficiente de variação:** compara o desvio-padrão com a média, permitindo avaliar a variabilidade proporcional entre subsistemas de escalas diferentes.

In [9]:
view_describe['variancia'] = view_describe['std'] ** 2
view_describe['amplitude'] = view_describe['max'] - view_describe['min']
view_describe['iqr'] = view_describe['75%'] - view_describe['25%']
view_describe['cv_%'] = view_describe['std'] / view_describe['mean'] * 100

view_describe.round(2)


,,count,mean,std,min,25%,50%,75%,max,variancia,amplitude,iqr,cv_%
id_subsistema,nom_subsistema,,,,,,,,,,,,
N,NORTE,227837.0,4592.68,1670.83,610.40,3275.08,4284.52,5640.59,10239.23,2791672.89,9628.83,2365.51,36.38
NE,NORDESTE,227837.0,8886.53,2478.98,149.86,6826.40,8768.52,10710.88,18156.97,6145341.84,18007.11,3884.48,27.90
S,SUL,227837.0,9737.16,2860.97,3615.83,7621.37,9392.76,11762.64,22737.44,8185149.34,19121.61,4141.27,29.38
SE,SUDESTE,227837.0,33826.25,7625.22,7970.58,28174.30,33508.33,39214.42,62149.88,58143980.05,54179.30,11040.12,22.54


As medidas absolutas de dispersão acompanham a magnitude das cargas. O SUDESTE apresenta a maior variância, amplitude e intervalo interquartil, mas também possui uma carga média muito superior à dos outros subsistemas.

O coeficiente de variação permite uma comparação proporcional. O NORTE apresentou a maior variação em relação à própria média, com 36,38%. Em seguida aparecem SUL, com 29,38%, NORDESTE, com 27,90%, e SUDESTE, com 22,54%.

Portanto, considerando todo o período histórico, o NORTE possui a maior dispersão relativa. O SUDESTE, embora apresente as maiores diferenças em valores absolutos, possui a menor variação proporcional à sua média.

Esses resultados ainda abrangem todos os anos em conjunto. Por isso, parte dessa variação pode estar relacionada ao crescimento da carga ao longo do tempo, e não apenas às oscilações horárias. Essa questão será analisada posteriormente ao comparar os anos separadamente.

Também chama atenção a carga mínima de 149,86 MWmed no NORDESTE, valor bastante distante do primeiro quartil de 6.826,40 MWmed. Neste momento, não podemos afirmar se o registro representa um erro ou uma ocorrência real. Ele será investigado posteriormente na análise das menores cargas.

## 4. Distribuição da Carga Elétrica

As medidas estatísticas resumem os dados em valores únicos, mas não mostram como as cargas estão distribuídas.

Nesta etapa, vamos observar separadamente a distribuição de cada subsistema. O objetivo é verificar onde os valores estão mais concentrados, se existe assimetria e se aparecem valores muito afastados da maior parte dos registros.

In [10]:
for subsystem in electric_data['nom_subsistema'].unique():
    subsystem_load = electric_data.loc[
        electric_data['nom_subsistema'] == subsystem,
        'val_cargaenergiahomwmed'
    ].dropna()

    frequency, limits = np.histogram(subsystem_load, bins=40)
    bin_centers = (limits[:-1] + limits[1:]) / 2
    bin_widths = limits[1:] - limits[:-1]

    fig = px.bar(
        x=bin_centers,
        y=frequency,
        title=f'Distribuição da carga elétrica - {subsystem}',
        color_discrete_sequence=['#636EFA']
    )

    fig.update_traces(
        width=bin_widths,
        marker_line_color='black',
        marker_line_width=0.6
    )
    fig.update_layout(
        xaxis_title='Carga elétrica (MWmed)',
        yaxis_title='Quantidade de registros',
        bargap=0.05
    )
    fig.show()


Os histogramas mostram que a carga elétrica possui distribuições diferentes entre os subsistemas.

No NORTE, existem várias faixas de concentração, principalmente entre aproximadamente 3.000 e 6.000 MWmed. Também aparece um grupo menor de registros com cargas mais elevadas, entre 7.000 e 9.000 MWmed.

O NORDESTE apresenta uma distribuição mais espalhada, com maior concentração entre aproximadamente 6.000 e 11.000 MWmed. O valor mínimo identificado anteriormente não fica visível no histograma, pois representa uma quantidade muito pequena diante do total de registros.

O SUL apresenta maior concentração entre 7.000 e 10.000 MWmed, com uma extensão em direção às cargas mais altas. O SUDESTE também possui uma faixa ampla, mas concentra a maior parte dos registros entre aproximadamente 25.000 e 42.000 MWmed.

A presença de diferentes concentrações pode estar relacionada às mudanças da carga ao longo dos 26 anos analisados. Portanto, os histogramas representam uma visão geral do histórico, mas não permitem afirmar que essas faixas ocorreram simultaneamente. Para compreender melhor esse comportamento, o próximo passo será analisar a evolução anual da carga.

## 5. Evolução Histórica da Carga por Subsistema

Os histogramas mostraram diferentes faixas de concentração nos dados. Como todos os anos foram analisados em conjunto, essas diferenças podem estar relacionadas à evolução da carga ao longo do tempo.

Nesta etapa, será calculada a carga média anual de cada subsistema. A média será utilizada porque queremos comparar o nível típico da carga entre os anos. A soma não seria adequada para essa primeira comparação, pois seria influenciada pela quantidade de horas registradas em cada ano.

In [11]:
electric_data['ano'] = electric_data['din_instante'].dt.year

annual_load = electric_data.groupby(['ano', 'nom_subsistema'])['val_cargaenergiahomwmed'].mean().unstack()
annual_load.round(2)

nom_subsistema,NORDESTE,NORTE,SUDESTE,SUL
ano,,,,
2000,5862.91,2486.96,25726.05,6736.47
2001,5293.86,2320.70,23266.03,6849.02
2002,5626.32,2339.39,25009.43,6692.67
2003,6040.62,2741.75,26143.64,6838.50
2004,6270.40,2973.58,27250.96,7236.77
2005,6694.96,3101.73,28359.73,7557.14
2006,6910.49,3353.17,29355.84,7851.72
2007,7241.27,3476.68,30845.26,8167.41
2008,7475.35,3630.11,31475.65,8427.00


In [12]:
fig = px.line(
    annual_load,
    x=annual_load.index,
    y=annual_load.columns,
    markers=True,
    title='Carga elétrica média anual por subsistema'
)

fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Carga média (MWmed)',
    legend_title='Subsistema',
    hovermode='x unified'
)
fig.show()


A carga elétrica média apresenta crescimento de longo prazo nos quatro subsistemas, embora esse movimento não tenha ocorrido de maneira contínua.

O SUDESTE permaneceu com a maior carga durante todo o período, passando de aproximadamente 25,7 mil MWmed em 2000 para 44,2 mil MWmed em 2025. O SUL apresentou a segunda maior carga, seguido pelo NORDESTE e pelo NORTE.

Também foram observados anos de redução ou estabilidade. Em 2020, por exemplo, NORDESTE, SUL e SUDESTE apresentaram queda na carga média, enquanto o NORTE permaneceu praticamente estável. Após esse período, as cargas voltaram a crescer.

Em 2025, NORTE, NORDESTE e SUL apresentaram crescimento em relação ao ano anterior, enquanto o SUDESTE registrou uma pequena redução. Neste momento, estamos apenas identificando esses movimentos. Os períodos que mais se destacarem serão investigados posteriormente.

Como o SUDESTE possui valores muito superiores aos demais, sua série ocupa grande parte da escala do gráfico e dificulta a comparação do crescimento proporcional. Por isso, será criada uma nova visualização na qual todos os subsistemas partirão do mesmo ponto de referência.

### 5.1 Crescimento Relativo por Subsistema

O gráfico anterior permite comparar os valores absolutos, mas não mostra com clareza qual subsistema apresentou o maior crescimento proporcional.

Para fazer essa comparação, o ano de 2000 será utilizado como base 100. Dessa forma:

- 100 representa o mesmo nível observado em 2000;
- 150 representa uma carga 50% maior;
- 200 representa o dobro da carga registrada no ano inicial.

Esse índice não representa a carga em MWmed. Ele mostra apenas a evolução proporcional de cada subsistema.

In [13]:
relative_growth = annual_load / annual_load.loc[2000] * 100

fig = px.line(
    relative_growth,
    x=relative_growth.index,
    y=relative_growth.columns,
    markers=True,
    title='Crescimento relativo da carga elétrica por subsistema'
)

fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Índice de crescimento — 2000 = 100',
    legend_title='Subsistema',
    hovermode='x unified'
)
fig.show()


In [14]:
growth_2000_2025 = (annual_load.loc[2025] / annual_load.loc[2000] - 1) * 100

growth_2000_2025.sort_values(ascending=False).round(2)


nom_subsistema
NORTE       234.21
NORDESTE    126.28
SUL         104.92
SUDESTE      71.91
dtype: float64

A análise proporcional mostra que o NORTE apresentou o maior crescimento entre 2000 e 2025. Sua carga média aumentou 234,21%, o que significa que em 2025 ela era aproximadamente 3,34 vezes o valor observado em 2000.

O NORDESTE apresentou crescimento de 126,28%, mais que dobrando sua carga média. O SUL também praticamente dobrou, com crescimento de 104,92%. Já o SUDESTE cresceu 71,91%.

Portanto, embora o SUDESTE ainda possua a maior carga em valores absolutos, ele apresentou o menor crescimento proporcional. O NORTE possui a menor carga absoluta, mas foi o subsistema que mais cresceu proporcionalmente.

Esse resultado também ajuda a interpretar o coeficiente de variação calculado anteriormente. Parte da maior dispersão relativa observada no NORTE pode estar relacionada à forte mudança no nível de sua carga ao longo dos anos, e não somente às oscilações de curto prazo.

Além do crescimento da demanda, a própria abrangência do sistema mudou durante o período. Segundo o ONS, o Estado do Amazonas foi integrado ao SIN em julho de 2013 pela interligação Tucuruí–Manaus–Macapá. Portanto, uma parcela do crescimento do NORTE representa a incorporação de uma carga que anteriormente não fazia parte do sistema interligado, e não apenas o aumento dos consumidores que já estavam presentes na série.

Fonte: [ONS — Interligação Tucuruí–Manaus–Macapá](https://www.ons.org.br/sites/multimidia/Documentos%20Compartilhados/dados/dados_relevantes_2013/HTML/01-07-destaques.html).

In [ ]:
hourly_load = electric_data.pivot(index='din_instante', columns='id_subsistema', values='val_cargaenergiahomwmed')

hourly_load.head()

id_subsistema,N,NE,S,SE
din_instante,,,,
2000-01-01 00:00:00,2373.7,5340.2,5777.0,21183.0
2000-01-01 01:00:00,2331.6,5180.8,5580.7,20409.9
2000-01-01 02:00:00,2332.9,5068.6,5098.7,19787.4
2000-01-01 03:00:00,2311.6,4905.1,4753.7,19089.0
2000-01-01 04:00:00,2287.8,4811.7,4584.1,18589.1


In [ ]:
hourly_load['carga_sin'] = hourly_load[['N', 'NE', 'S', 'SE']].sum(axis=1, min_count=4)
hourly_load.head()

id_subsistema,N,NE,S,SE,carga_sin
din_instante,,,,,
2000-01-01 00:00:00,2373.7,5340.2,5777.0,21183.0,34673.9
2000-01-01 01:00:00,2331.6,5180.8,5580.7,20409.9,33503.0
2000-01-01 02:00:00,2332.9,5068.6,5098.7,19787.4,32287.6
2000-01-01 03:00:00,2311.6,4905.1,4753.7,19089.0,31059.4
2000-01-01 04:00:00,2287.8,4811.7,4584.1,18589.1,30272.7


In [17]:
hourly_load['carga_sin'].isna().sum()

np.int64(77)

In [18]:
missing_sin = hourly_load[hourly_load['carga_sin'].isna()]
missing_sin.head()

id_subsistema,N,NE,S,SE,carga_sin
din_instante,,,,,
2013-12-01 00:00:00,NaN,NaN,NaN,NaN,NaN
2013-12-01 01:00:00,NaN,NaN,NaN,NaN,NaN
2013-12-01 02:00:00,NaN,NaN,NaN,NaN,NaN
2013-12-01 03:00:00,NaN,NaN,NaN,NaN,NaN
2013-12-01 04:00:00,NaN,NaN,NaN,NaN,NaN


In [19]:
missing_sin[['N', 'NE', 'S', 'SE']].isna().sum()

id_subsistema
N     77
NE    77
S     77
SE    77
dtype: int64

Os 77 horários apresentam valores ausentes nas quatro colunas de subsistema. Portanto, a carga total não deixou de ser calculada pela ausência de apenas um subsistema: nenhuma das quatro cargas ficou disponível nesses horários.

Entretanto, o resultado do `pivot()` não permite diferenciar um registro existente com carga nula de um registro completamente ausente da base. Antes de investigar essa diferença, vamos verificar como esses 77 horários estão distribuídos entre as datas.

In [ ]:
missing_sin_by_date = missing_sin.reset_index()

missing_sin_by_date['data'] = missing_sin_by_date['din_instante'].dt.date
missing_sin_by_date.groupby('data')['din_instante'].agg(['count', 'min', 'max'])

,count,min,max
data,,,
2013-12-01,24,2013-12-01,2013-12-01 23:00:00
2014-02-01,24,2014-02-01,2014-02-01 23:00:00
2014-10-19,1,2014-10-19,2014-10-19 00:00:00
2015-04-09,24,2015-04-09,2015-04-09 23:00:00
2015-10-18,1,2015-10-18,2015-10-18 00:00:00
2016-10-16,1,2016-10-16,2016-10-16 00:00:00
2017-10-15,1,2017-10-15,2017-10-15 00:00:00
2018-11-04,1,2018-11-04,2018-11-04 00:00:00


Os 77 horários estão concentrados em oito datas.

Em 01/12/2013, 01/02/2014 e 09/04/2015, a carga total não pôde ser calculada durante as 24 horas do dia, totalizando 72 horários.

Nas outras cinco datas, apenas o horário de 00:00 apresentou carga total ausente. Esses cinco registros, somados aos 72 anteriores, correspondem aos 77 horários identificados.

Quando a coluna `min` apresenta somente a data, ela também representa o horário de 00:00:00. O horário não é exibido porque todos os seus componentes são iguais a zero.

Ainda não sabemos se, nesses momentos, os registros dos quatro subsistemas existem com carga nula ou se algum registro está completamente ausente da base original.

In [ ]:
records_at_missing_times = electric_data[electric_data['din_instante'].isin(missing_sin.index)].copy()
records_at_missing_times['data'] = records_at_missing_times['din_instante'].dt.date
records_at_missing_times.groupby(['data', 'id_subsistema']).size().unstack(fill_value=0)

id_subsistema,N,NE,S,SE
data,,,,
2013-12-01,24,24,24,24
2014-02-01,0,24,24,24
2014-10-19,1,1,1,1
2015-04-09,0,24,24,24
2015-10-18,1,1,1,1
2016-10-16,1,1,1,1
2017-10-15,1,1,1,1
2018-11-04,1,1,1,1


A consulta mostrou que o subsistema NORTE não possui registros em 01/02/2014 e 09/04/2015. Para os outros subsistemas, existem 24 linhas em cada uma dessas datas.

Nas demais datas, os quatro subsistemas possuem registros nos horários analisados. Entretanto, ainda precisamos confirmar se todas essas linhas existentes estão com a carga nula.

In [22]:
records_at_missing_times.shape[0]

260

In [23]:
records_at_missing_times['val_cargaenergiahomwmed'].isna().sum()

np.int64(260)

Nos 77 horários sem carga total, seriam esperados 308 registros, considerando quatro subsistemas por horário.

A consulta encontrou 260 registros na base original e todos eles possuem carga nula. Os outros 48 registros não existem na base e correspondem às 24 horas do subsistema NORTE ausentes em 01/02/2014 e às 24 horas ausentes em 09/04/2015.

Portanto, os 77 horários sem carga total são resultado de duas situações:

- 260 registros existentes com carga nula;
- 48 registros completamente ausentes do subsistema NORTE.

Isso explica por que 260 valores nulos na base original resultaram em 77 horários sem carga total após a reorganização dos dados.

In [ ]:
expected_hours = pd.date_range(start=hourly_load.index.min(), end=hourly_load.index.max(), freq='h')
absent_hours = expected_hours.difference(hourly_load.index)

len(absent_hours)

14

In [25]:
absent_hours

DatetimeIndex(['2000-10-08', '2001-10-14', '2002-11-03', '2003-10-19',
               '2004-11-02', '2005-10-16', '2006-11-05', '2007-10-14',
               '2008-10-19', '2009-10-18', '2010-10-17', '2011-10-16',
               '2012-10-21', '2013-10-20'],
              dtype='datetime64[ns]', freq=None)

Os 14 horários completamente ausentes ocorrem sempre às 00:00, uma vez por ano, entre outubro e novembro de 2000 a 2013.

Após a comparação com as datas oficiais, foi confirmado que essas ocorrências coincidem com o início do horário de verão. Nesse momento, os relógios eram adiantados em uma hora a partir de 00:00. Por isso, na referência horária utilizada pela base, esse horário não chegou a ocorrer.

Também foi observada uma mudança na forma como essas transições aparecem nos dados. Entre 2000 e 2013, o horário de 00:00 não possui nenhuma linha. Entre 2014 e 2018, a linha passou a existir, mas sem uma carga válida.

Portanto, esses casos representam uma característica da transição de horário e não uma falha comum de coleta. Já os três dias completos sem carga permanecem classificados como falhas ou indisponibilidades da base, pois não estão relacionados ao início do horário de verão.

Fonte consultada: [Ministério de Minas e Energia — Horário de Verão](https://www.gov.br/mme/pt-br/assuntos/secretarias/secretaria-nacional-energia-eletrica/horario-de-verao).

In [26]:
valid_hours = hourly_load['carga_sin'].notna().sum()
unavailable_hours = hourly_load['carga_sin'].isna().sum() + len(absent_hours)
coverage_percentage = valid_hours / len(expected_hours) * 100

print(f'Horários esperados: {len(expected_hours):,}')
print(f'Horários com carga válida: {valid_hours:,}')
print(f'Horários indisponíveis: {unavailable_hours:,}')
print(f'Cobertura da série: {coverage_percentage:.3f}%')


Horários esperados: 227,928
Horários com carga válida: 227,837
Horários indisponíveis: 91
Cobertura da série: 99.960%


### 6.1 Cobertura da Série do SIN

Entre os 227.928 horários esperados no período, 227.837 possuem carga total válida. Foram identificados 91 horários indisponíveis, o que resulta em uma cobertura de 99,960%.

Os 91 horários indisponíveis são formados por:

- 72 horários pertencentes aos três dias completos sem carga válida;
- 19 horários relacionados ao início do horário de verão, sendo 14 completamente ausentes e 5 existentes com carga nula.

Portanto, apenas 0,040% dos horários esperados não possuem carga total válida.

Para as análises exploratórias anuais e mensais, esses valores serão mantidos como ausentes. Como representam uma parcela muito pequena da série, sua ausência não deve alterar de forma relevante as médias gerais. Entretanto, não será possível analisar o comportamento da carga especificamente nesses horários.

Na etapa de decomposição da série temporal, será necessário avaliar um tratamento para essas lacunas, pois esse método exige uma sequência temporal regular.

### 6.2 Estatísticas Descritivas da Carga Total

Após verificar a cobertura da série, serão analisadas as principais medidas estatísticas da carga total do SIN.

Essas medidas permitirão conhecer seu nível médio, sua dispersão e a faixa em que a maior parte dos registros está concentrada.

In [27]:
sin_statistics = hourly_load['carga_sin'].describe().round(2)

sin_statistics

count    227837.00
mean      57042.63
std       14056.38
min       22601.86
25%       46313.47
50%       56291.63
75%       67144.33
max      106148.66
Name: carga_sin, dtype: float64

A série do SIN possui 227.837 horários com carga total válida, resultado consistente com a verificação de cobertura realizada anteriormente.

A carga média do período foi de 57.042,63 MWmed, enquanto a mediana foi de 56.291,63 MWmed. A proximidade entre essas duas medidas indica que o centro da distribuição não é fortemente afetado pelos valores extremos, embora a média ligeiramente maior sugira alguma influência das cargas mais elevadas.

Metade dos registros está concentrada entre 46.313,47 MWmed e 67.144,33 MWmed, valores representados pelo primeiro e pelo terceiro quartil.

O desvio-padrão foi de 14.056,38 MWmed, mostrando que existe uma variação considerável ao redor da média histórica.

A menor carga total registrada foi de 22.601,86 MWmed e a maior foi de 106.148,66 MWmed. Essa diferença não deve ser interpretada imediatamente como presença de valores incorretos, pois a série reúne 26 anos de crescimento, além de variações sazonais e horárias. Os extremos serão investigados posteriormente junto com suas respectivas datas.

### 6.3 Evolução Anual da Carga Total

As estatísticas anteriores resumem todo o histórico em um único conjunto de valores. Para compreender como o nível da carga mudou, será calculada a média anual do SIN.

In [28]:
annual_sin = hourly_load.groupby(hourly_load.index.year)['carga_sin'].mean()
annual_sin.index.name = 'ano'

annual_sin.round(2)


ano
2000    40812.38
2001    37729.60
2002    39667.80
2003    41764.51
2004    43731.71
2005    45713.57
2006    47471.22
2007    49730.61
2008    51008.11
2009    50598.08
2010    54061.77
2011    56029.85
2012    58100.14
2013    58744.33
2014    61555.06
2015    61353.78
2016    61551.74
2017    62681.42
2018    63255.71
2019    64581.11
2020    63408.47
2021    68525.44
2022    68797.20
2023    73704.99
2024    78943.25
2025    79608.91
Name: carga_sin, dtype: float64

In [29]:
fig = px.line(
    x=annual_sin.index,
    y=annual_sin.values,
    markers=True,
    title='Carga elétrica média anual do SIN'
)

fig.update_traces(line_color='#636EFA')
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Carga média (MWmed)',
    hovermode='x unified',
    showlegend=False
)
fig.show()


A carga média anual do SIN apresenta uma trajetória de crescimento no longo prazo. O nível observado em 2025 é quase o dobro do registrado no início da série.

Esse crescimento não ocorreu de maneira constante. Em 2001 houve uma redução acentuada, seguida por uma recuperação gradual nos anos posteriores. Também aparecem pequenas quedas em 2009, períodos de estabilidade entre 2014 e 2016 e uma nova redução em 2020.

A partir de 2021, a carga voltou a crescer. O avanço ficou mais intenso entre 2022 e 2024, enquanto 2025 apresentou uma variação menor em relação ao ano anterior.

Essa evolução ajuda a explicar a amplitude observada nas estatísticas descritivas e as diferentes concentrações dos histogramas. Os registros dos primeiros anos pertencem a um nível de carga muito menor que o observado nos anos mais recentes.

O gráfico permite identificar esses movimentos, mas ainda não mostra o tamanho de cada variação. Por isso, o próximo passo será calcular a mudança percentual entre anos consecutivos.

### 6.4 Variação Percentual Anual

Para medir a intensidade das mudanças observadas no gráfico, será calculada a variação percentual da carga média em relação ao ano anterior.

Valores positivos representam crescimento e valores negativos representam redução.

In [30]:
annual_variation = annual_sin.pct_change() * 100

annual_variation.round(2)

ano
2000     NaN
2001   -7.55
2002    5.14
2003    5.29
2004    4.71
2005    4.53
2006    3.84
2007    4.76
2008    2.57
2009   -0.80
2010    6.85
2011    3.64
2012    3.69
2013    1.11
2014    4.78
2015   -0.33
2016    0.32
2017    1.84
2018    0.92
2019    2.10
2020   -1.82
2021    8.07
2022    0.40
2023    7.13
2024    7.11
2025    0.84
Name: carga_sin, dtype: float64

In [31]:
annual_variation_data = annual_variation.dropna()

fig = px.bar(
    x=annual_variation_data.index,
    y=annual_variation_data.values,
    title='Variação anual da carga média do SIN'
)

fig.update_traces(marker_color='#636EFA')
fig.add_hline(y=0, line_color='black')
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Variação em relação ao ano anterior (%)',
    showlegend=False
)
fig.show()


A carga média do SIN cresceu na maior parte do período. Dos 25 anos com possibilidade de comparação, apenas quatro apresentaram redução.

A maior queda ocorreu em 2001, quando a carga média diminuiu 7,55%. Esse período coincide com o racionamento de energia ocorrido no país. Portanto, o contexto histórico ajuda a explicar a redução observada, embora o dataset sozinho não seja suficiente para estabelecer essa relação como causa.

Em 2020, a carga caiu 1,82%. O ONS registrou naquele ano impactos das restrições adotadas durante a pandemia sobre a carga do sistema, principalmente nos meses de maior interrupção das atividades econômicas.

O maior crescimento aconteceu em 2021, com aumento de 8,07%. Esse resultado deve ser interpretado com cuidado, pois reúne a recuperação em relação ao nível mais baixo de 2020 e uma mudança na composição dos dados divulgados pelo ONS.

Outros crescimentos elevados ocorreram em 2023 e 2024, ambos superiores a 7%. Parte desse movimento também pode estar relacionada à incorporação da estimativa de micro e minigeração distribuída aos dados de carga a partir de abril de 2023.

Dessa forma, as variações recentes não devem ser interpretadas somente como aumento real da demanda. Parte da diferença também pode estar relacionada às mudanças metodológicas realizadas pelo ONS.

Fontes consultadas:

- [EPE — PNE 2030: Análise Retrospectiva](https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/publicacoes/PublicacoesArquivos/publicacao-165/topico-173/PNE%202030%20-%20An%C3%A1lise%20Retrospectiva.pdf);
- [ONS — Boletim Mensal de Carga de abril de 2020](https://www.ons.org.br/AcervoDigitalDocumentosEPublicacoes/BoletimMensalCarga_abril-2020%20%28002%29.pdf);
- [ONS — Histórico da Carga de Energia](https://www.ons.org.br/Paginas/resultados-da-operacao/historico-da-operacao/carga_energia.aspx).

In [32]:
total_growth_sin = (annual_sin.loc[2025] / annual_sin.loc[2000] - 1) * 100
average_annual_growth = ((annual_sin.loc[2025] / annual_sin.loc[2000]) ** (1 / 25) - 1) * 100

print(f'Crescimento entre 2000 e 2025: {total_growth_sin:.2f}%')
print(f'Crescimento médio anual: {average_annual_growth:.2f}%')


Crescimento entre 2000 e 2025: 95.06%
Crescimento médio anual: 2.71%


Entre 2000 e 2025, a carga média anual do SIN cresceu 95,06%.

A taxa média anual composta foi de 2,71%. Isso significa que, se o crescimento tivesse ocorrido de forma constante, a carga teria aumentado aproximadamente 2,71% ao ano para sair do nível de 2000 e chegar ao valor de 2025.

Essa taxa não significa que todos os anos cresceram 2,71%. Como observado anteriormente, houve anos de queda, estabilidade e crescimento mais intenso.

O resultado resume a expansão da carga no longo prazo, mas deve ser interpretado considerando as mudanças metodológicas realizadas pelo ONS em 2021 e 2023.

## 7. Participação dos Subsistemas na Carga do SIN

Os subsistemas cresceram em ritmos diferentes. Por isso, será analisada a participação percentual de cada um na carga total do SIN.

O cálculo será realizado em cada horário, dividindo a carga do subsistema pela carga total registrada no mesmo instante. Depois, será calculada a participação média de cada ano.

In [33]:
subsystems = ['N', 'NE', 'S', 'SE']

hourly_share = hourly_load[subsystems].copy()

In [34]:
for subsystem in subsystems:
    hourly_share[subsystem] = hourly_share[subsystem] / hourly_load['carga_sin'] * 100


In [35]:
hourly_share.head()

id_subsistema,N,NE,S,SE
din_instante,,,,
2000-01-01 00:00:00,6.845783,15.401210,16.660947,61.092061
2000-01-01 01:00:00,6.959377,15.463690,16.657314,60.919619
2000-01-01 02:00:00,7.225374,15.698287,15.791511,61.284828
2000-01-01 03:00:00,7.442513,15.792642,15.305189,61.459655
2000-01-01 04:00:00,7.557304,15.894519,15.142686,61.405491


In [36]:
annual_share = hourly_share.groupby(hourly_share.index.year).mean()
annual_share.index.name = 'ano'

annual_share.round(2)


id_subsistema,N,NE,S,SE
ano,,,,
2000,6.18,14.43,16.41,62.98
2001,6.26,14.09,18.15,61.50
2002,5.98,14.25,16.79,62.98
2003,6.65,14.55,16.27,62.53
2004,6.89,14.41,16.44,62.25
2005,6.88,14.73,16.43,61.97
2006,7.16,14.63,16.44,61.76
2007,7.08,14.63,16.31,61.97
2008,7.21,14.73,16.41,61.64


In [37]:
share_variation = annual_share - annual_share.loc[2000]

fig = px.line(
    share_variation,
    x=share_variation.index,
    y=share_variation.columns,
    markers=True,
    title='Mudança na participação dos subsistemas em relação a 2000'
)

fig.add_hline(y=0, line_color='black')
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Variação da participação (pontos percentuais)',
    legend_title='Subsistema',
    hovermode='x unified'
)
fig.show()


O gráfico utiliza 2000 como ponto de referência. Valores acima de zero representam ganho de participação e valores abaixo de zero representam perda de participação na carga total do SIN.

Para medir essas mudanças, serão comparados os valores do primeiro e do último ano da série.

In [38]:
share_comparison = annual_share.loc[[2000, 2025]].T.copy()

share_comparison.columns = ['participacao_2000', 'participacao_2025']
share_comparison['variacao_pp'] = (
    share_comparison['participacao_2025'] - share_comparison['participacao_2000']
)

share_comparison.round(2)


,participacao_2000,participacao_2025,variacao_pp
id_subsistema,,,
N,6.18,10.57,4.39
NE,14.43,16.78,2.35
S,16.41,17.20,0.80
SE,62.98,55.44,-7.53


O SUDESTE permaneceu como o principal componente da carga do SIN, mas sua participação média diminuiu de 62,98% em 2000 para 55,44% em 2025, uma redução de 7,53 pontos percentuais.

O NORTE apresentou o maior ganho, passando de 6,18% para 10,57%, aumento de 4,39 pontos percentuais. O NORDESTE ganhou 2,35 pontos e o SUL, 0,80 ponto percentual.

A redução da participação do SUDESTE não significa que sua carga tenha diminuído. A análise anual mostrou que sua carga cresceu em valores absolutos, mas em ritmo proporcional menor que a dos demais subsistemas.

## 8. Sazonalidade Mensal da Carga

Depois de analisar a tendência anual, vamos reduzir a série para médias mensais. Essa visualização permite observar se existem oscilações que se repetem dentro dos anos sem o excesso de variação presente nos dados horários.

In [39]:
monthly_sin = hourly_load['carga_sin'].resample('ME').mean()

monthly_sin.head()

din_instante
2000-01-31    39253.390659
2000-02-29    40665.397845
2000-03-31    40556.063720
2000-04-30    40214.947917
2000-05-31    40408.382258
Freq: ME, Name: carga_sin, dtype: float64

In [40]:
fig = px.line(
    x=monthly_sin.index,
    y=monthly_sin.values,
    title='Carga elétrica média mensal do SIN'
)

fig.update_traces(line_color='#636EFA')
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Carga média (MWmed)',
    hovermode='x unified',
    showlegend=False
)
fig.show()


A série mensal mantém a tendência de crescimento observada anteriormente e também apresenta oscilações recorrentes dentro dos anos.

Entretanto, comparar diretamente os meses em MWmed pode misturar dois efeitos: a sazonalidade e o crescimento de longo prazo. Um mês dos anos recentes tende a apresentar carga maior que o mesmo mês dos primeiros anos simplesmente porque o nível da série aumentou.

Para separar melhor esses efeitos, cada média mensal será comparada com a média do próprio ano. O resultado será um índice no qual 100 representa o nível médio anual.

In [41]:
monthly_analysis = monthly_sin.to_frame(name='carga_media')

monthly_analysis['ano'] = monthly_analysis.index.year
monthly_analysis['mes'] = monthly_analysis.index.month
monthly_analysis['media_anual'] = monthly_analysis.groupby('ano')['carga_media'].transform('mean')
monthly_analysis['indice_sazonal'] = (
    monthly_analysis['carga_media'] / monthly_analysis['media_anual'] * 100
)

month_names = {
    1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr',
    5: 'Mai', 6: 'Jun', 7: 'Jul', 8: 'Ago',
    9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'
}

monthly_seasonality = monthly_analysis.groupby('mes')['indice_sazonal'].mean()
monthly_seasonality.index = monthly_seasonality.index.map(month_names)

monthly_seasonality.round(2)


mes
Jan    101.71
Fev    104.60
Mar    104.45
Abr    100.20
Mai     96.98
Jun     95.15
Jul     95.03
Ago     97.35
Set     99.61
Out    101.40
Nov    101.69
Dez    101.84
Name: indice_sazonal, dtype: float64

In [42]:
fig = px.line(
    x=monthly_seasonality.index,
    y=monthly_seasonality.values,
    markers=True,
    title='Perfil mensal médio da carga do SIN'
)

fig.update_traces(line_color='#636EFA')
fig.add_hline(y=100, line_color='black')
fig.update_layout(
    xaxis_title='Mês',
    yaxis_title='Índice sazonal — média anual = 100',
    showlegend=False
)
fig.show()


O perfil médio apresenta cargas mais elevadas em fevereiro e março, com índices de 104,60 e 104,45. Isso significa que, em média, esses meses ficaram aproximadamente 4,6% e 4,5% acima do nível médio de seus respectivos anos.

As menores cargas aparecem em junho e julho, com índices próximos de 95. Portanto, esses meses ficaram cerca de 5% abaixo da média anual.

Entre outubro e dezembro, a carga volta a permanecer acima da referência de 100. O resultado confirma a existência de sazonalidade mensal, mas representa o comportamento médio dos 26 anos. Anos específicos podem apresentar perfis diferentes.

### 8.1 Sazonalidade Mensal por Subsistema

O comportamento geral do SIN pode esconder diferenças regionais. Para compará-las, o mesmo índice será calculado separadamente para cada subsistema.

In [ ]:
subsystem_monthly = (
    electric_data.assign(mes=electric_data['din_instante'].dt.month)
    .groupby(['ano', 'mes', 'nom_subsistema'])['val_cargaenergiahomwmed']
    .mean()
    .reset_index()
)

subsystem_monthly['media_anual'] = (subsystem_monthly.groupby(['ano', 'nom_subsistema'])['val_cargaenergiahomwmed'].transform('mean'))
subsystem_monthly['indice_sazonal'] = (subsystem_monthly['val_cargaenergiahomwmed'] / subsystem_monthly['media_anual'] * 100)

subsystem_seasonality = (subsystem_monthly.groupby(['mes', 'nom_subsistema'])['indice_sazonal'].mean().unstack())
subsystem_seasonality.index = subsystem_seasonality.index.map(month_names)
subsystem_seasonality.round(2)

nom_subsistema,NORDESTE,NORTE,SUDESTE,SUL
mes,,,,
Jan,99.97,95.80,102.10,105.08
Fev,100.98,96.84,105.42,108.95
Mar,101.84,98.34,105.47,106.32
Abr,100.23,98.93,100.71,99.00
Mai,99.02,99.89,96.55,95.26
Jun,95.37,99.17,94.47,95.39
Jul,94.87,98.68,94.23,96.07
Ago,96.79,102.02,97.04,96.47
Set,99.81,103.46,100.07,95.88


In [44]:
fig = px.line(
    subsystem_seasonality,
    x=subsystem_seasonality.index,
    y=subsystem_seasonality.columns,
    markers=True,
    title='Perfil mensal médio da carga por subsistema'
)

fig.add_hline(y=100, line_color='black')
fig.update_layout(
    xaxis_title='Mês',
    yaxis_title='Índice sazonal — média anual = 100',
    legend_title='Subsistema',
    hovermode='x unified'
)
fig.show()


Os subsistemas não apresentam o mesmo comportamento mensal.

SUL e SUDESTE possuem as maiores elevações em fevereiro e março e ficam abaixo da média principalmente entre maio e agosto. O NORDESTE apresenta menor carga em junho e julho e seus maiores índices entre outubro e dezembro.

O NORTE possui um padrão diferente: janeiro e fevereiro ficam abaixo de sua média anual, enquanto os maiores índices aparecem entre agosto e novembro.

Portanto, a sazonalidade observada no SIN resulta da combinação de perfis regionais distintos. Os dados mostram quando essas diferenças ocorrem, mas não permitem determinar suas causas sem acrescentar informações como temperatura, precipitação, calendário e atividade econômica.

## 9. Comportamento ao Longo da Semana

Para verificar a existência de um ciclo semanal, será calculada inicialmente a carga média de cada dia. Em seguida, cada valor será comparado com a média do respectivo ano, evitando que o crescimento histórico domine a comparação.

In [45]:
daily_sin = hourly_load['carga_sin'].resample('D').mean().to_frame('carga_media')

daily_sin['ano'] = daily_sin.index.year
daily_sin['dia_semana'] = daily_sin.index.dayofweek
daily_sin['media_anual'] = daily_sin.groupby('ano')['carga_media'].transform('mean')
daily_sin['indice_anual'] = daily_sin['carga_media'] / daily_sin['media_anual'] * 100

day_names = {
    0: 'Segunda', 1: 'Terça', 2: 'Quarta',
    3: 'Quinta', 4: 'Sexta', 5: 'Sábado', 6: 'Domingo'
}

weekly_profile = daily_sin.groupby('dia_semana')['indice_anual'].mean()
weekly_profile.index = weekly_profile.index.map(day_names)

weekly_profile.round(2)


dia_semana
Segunda    101.51
Terça      104.00
Quarta     104.48
Quinta     104.37
Sexta      103.75
Sábado      95.57
Domingo     86.33
Name: indice_anual, dtype: float64

In [46]:
fig = px.line(
    x=weekly_profile.index,
    y=weekly_profile.values,
    markers=True,
    title='Carga média do SIN por dia da semana'
)

fig.update_traces(line_color='#636EFA')
fig.add_hline(y=100, line_color='black')
fig.update_layout(
    xaxis_title='Dia da semana',
    yaxis_title='Índice — média anual = 100',
    showlegend=False
)
fig.show()


A carga apresenta um ciclo semanal bem definido. Os maiores índices ocorrem de terça a quinta-feira, todos próximos de 104. A segunda-feira fica mais próxima da média anual, com índice de 101,51.

No fim de semana, a carga diminui. O sábado apresenta índice de 95,57 e o domingo, de 86,33, sendo o dia de menor carga média.

Esses resultados indicam que o dia da semana deve ser considerado em uma futura modelagem da série. Também será importante incluir feriados, que não podem ser identificados apenas com a data e o dia da semana.

## 10. Comportamento ao Longo do Dia

Depois de confirmar o ciclo semanal, vamos analisar as 24 horas do dia. Dias úteis e finais de semana serão separados para verificar se a curva horária também muda entre esses dois grupos.

Novamente, as cargas serão divididas pela média do próprio ano. Dessa forma, o gráfico compara o formato das curvas sem ser dominado pelo crescimento de longo prazo.

In [47]:
hourly_profile = hourly_load[['carga_sin']].dropna().copy()

hourly_profile['ano'] = hourly_profile.index.year
hourly_profile['hora'] = hourly_profile.index.hour
hourly_profile['dia_semana'] = hourly_profile.index.dayofweek
hourly_profile['tipo_dia'] = 'Dia útil'

hourly_profile.loc[hourly_profile['dia_semana'] >= 5, 'tipo_dia'] = 'Fim de semana'

hourly_profile['media_anual'] = hourly_profile.groupby('ano')['carga_sin'].transform('mean')
hourly_profile['indice_anual'] = hourly_profile['carga_sin'] / hourly_profile['media_anual'] * 100

typical_hour = hourly_profile.groupby(['hora', 'tipo_dia'])['indice_anual'].mean().unstack()

typical_hour.round(2)


tipo_dia,Dia útil,Fim de semana
hora,,
0,93.27,92.35
1,88.42,87.58
2,85.71,84.41
3,84.51,82.57
4,84.65,81.71
5,86.82,81.10
6,90.66,79.49
7,96.37,80.55
8,104.00,84.51


In [48]:
fig = px.line(
    typical_hour,
    x=typical_hour.index,
    y=typical_hour.columns,
    markers=True,
    title='Perfil horário da carga do SIN por tipo de dia'
)

fig.add_hline(y=100, line_color='black')
fig.update_layout(
    xaxis_title='Hora',
    yaxis_title='Índice — média anual = 100',
    legend_title='Tipo de dia',
    hovermode='x unified'
)
fig.update_xaxes(dtick=1)
fig.show()


Nos dias úteis, a carga diminui durante a madrugada e atinge seu menor índice por volta das 03:00. A partir das 06:00, ocorre uma elevação rápida, e a carga permanece acima da média principalmente entre 08:00 e 22:00. O maior índice médio aparece às 19:00.

Nos finais de semana, a carga permanece abaixo da média anual durante quase todo o dia. A diferença em relação aos dias úteis é maior no período comercial, enquanto as duas curvas se aproximam à noite. O pico médio do fim de semana também ocorre às 19:00.

O resultado confirma que hora e tipo de dia estão associados ao nível da carga. Para observar se o formato também varia regionalmente, o perfil horário será calculado por subsistema.

### 10.1 Perfil Horário por Subsistema

In [ ]:
subsystem_hourly = electric_data.dropna(subset=['val_cargaenergiahomwmed']).copy()

subsystem_hourly['hora'] = subsystem_hourly['din_instante'].dt.hour
subsystem_hourly['media_anual'] = (subsystem_hourly.groupby(['ano', 'nom_subsistema'])['val_cargaenergiahomwmed'].transform('mean'))
subsystem_hourly['indice_anual'] = (subsystem_hourly['val_cargaenergiahomwmed'] / subsystem_hourly['media_anual'] * 100)

hourly_by_subsystem = (subsystem_hourly.groupby(['hora', 'nom_subsistema'])['indice_anual'].mean().unstack())

hourly_by_subsystem.round(2)


nom_subsistema,NORDESTE,NORTE,SUDESTE,SUL
hora,,,,
0,98.89,102.45,91.51,88.45
1,95.49,100.34,86.24,82.69
2,93.30,98.61,83.22,79.37
3,91.95,97.18,81.82,78.07
4,91.27,96.09,81.90,78.08
5,88.87,95.21,84.19,80.75
6,85.63,92.10,87.64,86.40
7,89.29,91.06,92.04,93.90
8,95.74,94.48,98.51,102.43


In [50]:
fig = px.line(
    hourly_by_subsystem,
    x=hourly_by_subsystem.index,
    y=hourly_by_subsystem.columns,
    markers=True,
    title='Perfil horário médio da carga por subsistema'
)

fig.add_hline(y=100, line_color='black')
fig.update_layout(
    xaxis_title='Hora',
    yaxis_title='Índice — média anual = 100',
    legend_title='Subsistema',
    hovermode='x unified'
)
fig.update_xaxes(dtick=1)
fig.show()


Os quatro subsistemas possuem redução durante a madrugada e elevação ao longo do dia, mas a intensidade e o horário dessas mudanças são diferentes.

SUL e SUDESTE apresentam as maiores variações dentro do dia. Ambos atingem seus menores índices entre 03:00 e 04:00 e os maiores por volta das 19:00. O NORDESTE alcança seu menor índice às 06:00 e o maior às 19:00.

O NORTE possui a curva mais estável. Seu menor índice médio aparece às 07:00 e o maior ocorre mais tarde, às 21:00. Além disso, sua carga permanece relativamente elevada durante a madrugada quando comparada com sua própria média anual.

Essas diferenças reforçam que um único perfil do SIN não representa perfeitamente todos os subsistemas.

## 11. Variabilidade da Carga ao Longo dos Anos

O coeficiente de variação calculado no início utilizou todos os anos em conjunto. Agora ele será calculado separadamente para cada ano e subsistema.

Essa medida divide o desvio-padrão pela média e permite verificar se a oscilação proporcional da carga mudou ao longo do tempo.

In [ ]:
annual_variability = (electric_data.groupby(['ano', 'nom_subsistema'])['val_cargaenergiahomwmed'].agg(['mean', 'std']))

annual_variability['cv_%'] = annual_variability['std'] / annual_variability['mean'] * 100
annual_cv = annual_variability['cv_%'].unstack()

annual_cv.round(2)


nom_subsistema,NORDESTE,NORTE,SUDESTE,SUL
ano,,,,
2000,12.79,6.75,15.31,18.92
2001,17.12,11.54,19.66,19.57
2002,12.53,8.79,15.02,18.69
2003,11.28,6.87,14.67,18.87
2004,12.07,6.02,14.81,18.91
2005,11.43,6.45,14.94,18.90
2006,11.74,6.21,15.02,18.78
2007,11.69,6.29,14.63,19.16
2008,11.52,6.36,14.71,19.20


In [52]:
fig = px.line(
    annual_cv,
    x=annual_cv.index,
    y=annual_cv.columns,
    markers=True,
    title='Coeficiente de variação anual por subsistema'
)

fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Coeficiente de variação (%)',
    legend_title='Subsistema',
    hovermode='x unified'
)
fig.show()


O SUL apresentou a maior variabilidade proporcional na maior parte do período, geralmente entre 18% e 21%. O SUDESTE aparece em seguida, enquanto NORDESTE e NORTE possuem coeficientes menores na maioria dos anos.

Não existe uma tendência contínua de aumento ou redução da variabilidade. Alguns anos apresentam elevações pontuais, como 2001 em vários subsistemas, 2013 no NORTE e 2014 no SUL.

O valor do NORTE em 2013 chama atenção porque é muito superior aos anos vizinhos. Antes de interpretar essa diferença como maior oscilação horária, vamos observar suas médias mensais naquele ano.

In [ ]:
north_2013 = subsystem_monthly[(subsystem_monthly['ano'] == 2013) & (subsystem_monthly['nom_subsistema'] == 'NORTE')][['mes', 'val_cargaenergiahomwmed']]

north_2013.round(2)

,mes,val_cargaenergiahomwmed
625,1,3955.97
629,2,4068.56
633,3,4218.49
637,4,4244.86
641,5,4215.81
645,6,4017.35
649,7,4637.35
653,8,5179.06
657,9,5188.62
661,10,5141.88


A média mensal do NORTE muda de nível a partir de julho de 2013. Em junho, a carga média era de aproximadamente 4.017 MWmed; em julho passou para 4.637 MWmed e, em agosto, para 5.179 MWmed.

Esse ponto coincide com a integração do Amazonas ao SIN, identificada anteriormente por meio da documentação do ONS. Como o ano de 2013 reúne meses anteriores e posteriores à mudança de abrangência, seu coeficiente de variação ficou elevado.

Portanto, nesse caso, o aumento da dispersão não representa apenas maior oscilação da demanda: ele também reflete uma mudança estrutural na composição da série.

## 12. Análise dos Valores Extremos

As estatísticas descritivas mostraram uma grande distância entre as cargas mínima e máxima do SIN. Para entender esses valores, vamos identificar inicialmente os dez maiores horários registrados.

In [54]:
largest_sin_loads = hourly_load['carga_sin'].nlargest(10).to_frame()

largest_sin_loads.round(2)


,carga_sin
din_instante,
2025-02-26 14:00:00,106148.66
2025-02-25 14:00:00,105314.79
2025-02-25 15:00:00,105184.59
2025-02-26 13:00:00,105152.44
2025-02-24 14:00:00,105123.00
2025-02-26 15:00:00,105092.43
2025-02-25 13:00:00,105036.29
2025-02-27 14:00:00,104895.86
2025-02-24 15:00:00,104721.93


Os dez maiores registros estão concentrados em fevereiro de 2025, principalmente entre 13:00 e 15:00. O maior valor ocorreu em 26/02/2025 às 14:00, quando a carga total alcançou 106.148,66 MWmed.

Essa concentração é coerente com dois comportamentos observados anteriormente: fevereiro apresentou um dos maiores índices sazonais e o período da tarde possui cargas elevadas nos dias úteis. A análise, entretanto, não permite determinar sozinha quais condições específicas produziram esses picos.

Agora serão observados os dez menores valores da carga total.

In [55]:
smallest_sin_loads = hourly_load['carga_sin'].nsmallest(10).to_frame()

smallest_sin_loads.round(2)


,carga_sin
din_instante,
2001-12-25 07:00:00,22601.86
2001-07-29 07:00:00,22677.34
2002-01-01 07:00:00,22987.90
2001-07-29 06:00:00,23234.51
2001-12-25 06:00:00,23295.70
2001-07-01 07:00:00,23301.62
2001-07-15 07:00:00,23348.83
2002-01-01 08:00:00,23439.33
2001-12-25 08:00:00,23443.80


Os menores valores do SIN estão concentrados em 2001 e no início de 2002, período em que o nível geral da série era muito inferior ao atual.

Vários registros ocorreram entre 05:00 e 08:00, faixa identificada anteriormente como período de baixa carga. Também aparecem o Natal e o Ano-Novo, além de domingos de julho de 2001. Esses elementos ajudam a contextualizar os valores baixos, mas não substituem uma variável de feriado em análises futuras.

A menor carga total foi registrada em 25/12/2001 às 07:00, com 22.601,86 MWmed.

### 12.1 Menor Valor de Cada Subsistema

No panorama estatístico, o mínimo de 149,86 MWmed do NORDESTE se destacou pela distância em relação aos demais valores. Vamos localizar os mínimos dos quatro subsistemas antes de analisá-los.

In [56]:
minimum_records = electric_data.loc[
    electric_data
    .groupby('id_subsistema')['val_cargaenergiahomwmed']
    .idxmin(),
    [
        'id_subsistema',
        'nom_subsistema',
        'din_instante',
        'val_cargaenergiahomwmed'
    ]
].sort_values('id_subsistema')

minimum_records

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
41448,N,NORTE,2001-03-07 19:00:00,610.400000
449425,NE,NORDESTE,2012-10-26 01:00:00,149.860000
104562,S,SUL,2002-12-25 07:00:00,3615.830005
72147,SE,SUDESTE,2002-01-21 14:00:00,7970.579997


Os mínimos ocorreram em datas e horários diferentes. Apenas a tabela não permite saber se eles fazem parte de uma queda gradual ou se representam reduções abruptas.

Para fazer essa distinção, será visualizada uma janela de quatro horas antes e quatro horas depois de cada registro mínimo.

In [57]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=minimum_records['nom_subsistema'].tolist(),
    vertical_spacing=0.15
)

for position, (_, record) in enumerate(minimum_records.iterrows()):
    row = position // 2 + 1
    column = position % 2 + 1
    start_time = record['din_instante'] - pd.Timedelta(hours=4)
    end_time = record['din_instante'] + pd.Timedelta(hours=4)

    context = electric_data[
        (electric_data['id_subsistema'] == record['id_subsistema'])
        & (electric_data['din_instante'].between(start_time, end_time))
    ]

    fig.add_trace(
        go.Scatter(
            x=context['din_instante'],
            y=context['val_cargaenergiahomwmed'],
            mode='lines+markers',
            name=record['nom_subsistema'],
            showlegend=False
        ),
        row=row,
        col=column
    )

    fig.update_yaxes(title_text='Carga (MWmed)', row=row, col=column)

fig.update_layout(
    title='Comportamento ao redor da menor carga de cada subsistema',
    height=700,
    hovermode='x unified'
)
fig.show()


O mínimo do SUL faz parte de uma curva relativamente gradual durante a manhã de 25/12/2002. Já NORTE, NORDESTE e SUDESTE apresentam reduções muito mais abruptas em relação às horas vizinhas.

No NORDESTE, a queda não está limitada a um único registro: a carga permanece muito baixa entre 00:00 e 03:00 de 26/10/2012 antes de retornar ao nível anterior. No SUDESTE, a redução também se estende por algumas horas em 21/01/2002. No NORTE, o menor valor aparece em uma queda mais curta em 07/03/2001.

Os dados confirmam que esses registros são atípicos em relação ao comportamento local, mas não permitem afirmar se representam falhas de medição, indisponibilidades ou acontecimentos reais do sistema. Por isso, eles serão preservados na análise exploratória e sinalizados para uma decisão de tratamento antes da modelagem.

## 13. Relação Entre os Subsistemas

Como todos os subsistemas apresentaram tendência de crescimento, calcular a correlação diretamente sobre os níveis horários poderia produzir valores elevados apenas porque as séries crescem ao longo do tempo.

Para reduzir esse efeito, será calculada a variação percentual das médias diárias. Em seguida, será medida a correlação entre essas variações.

In [58]:
daily_subsystem_load = hourly_load[subsystems].resample('D').mean()
daily_subsystem_variation = daily_subsystem_load.pct_change(fill_method=None) * 100

subsystem_correlation = daily_subsystem_variation.corr()

subsystem_correlation.round(3)

id_subsistema,N,NE,S,SE
id_subsistema,,,,
N,1.000,0.863,0.888,0.890
NE,0.863,1.000,0.913,0.915
S,0.888,0.913,1.000,0.965
SE,0.890,0.915,0.965,1.000


Todas as correlações são positivas e superiores a 0,86. Isso indica que as variações diárias dos subsistemas costumam ocorrer na mesma direção, embora não sejam idênticas.

A relação mais forte aparece entre SUL e SUDESTE, com correlação de 0,965. A menor aparece entre NORTE e NORDESTE, com 0,863, ainda considerada elevada dentro deste conjunto de dados.

Correlação não representa causalidade. O resultado mostra apenas que aumentos e reduções diárias tendem a ocorrer de forma semelhante entre os subsistemas.

## 14. Síntese da Análise Exploratória

A análise mostrou que a carga elétrica possui tendência de longo prazo, sazonalidade mensal, ciclo semanal e um padrão horário bem definido. Os principais resultados foram:

- A carga média do SIN cresceu 95,06% entre 2000 e 2025, equivalente a uma taxa média composta de 2,71% ao ano;
- O SUDESTE continuou com a maior carga absoluta, mas sua participação no SIN diminuiu de 62,98% para 55,44%;
- O NORTE apresentou o maior crescimento proporcional e ganhou 4,39 pontos percentuais de participação. Parte dessa mudança está relacionada à integração do Amazonas ao SIN em julho de 2013;
- Fevereiro e março apresentaram as maiores cargas mensais relativas do SIN, enquanto junho e julho apresentaram as menores;
- Os perfis mensais diferem entre os subsistemas, principalmente no NORTE, que possui comportamento distinto dos demais;
- A carga é maior entre terça e quinta-feira e menor aos domingos;
- Nos dias úteis, a carga aumenta rapidamente pela manhã e permanece elevada durante a tarde e o início da noite. Nos finais de semana, ela fica abaixo da média anual durante quase todo o dia;
- Os perfis horários também variam entre os subsistemas. SUL e SUDESTE possuem oscilações diárias mais intensas, enquanto o NORTE apresenta uma curva mais estável;
- Os maiores registros do SIN ocorreram em fevereiro de 2025 no período da tarde. Os menores se concentraram nos primeiros anos, durante a madrugada, em domingos e feriados;
- Três mínimos regionais apresentaram quedas abruptas em relação às horas vizinhas e deverão ser avaliados antes da modelagem;
- As variações diárias dos subsistemas possuem correlações positivas elevadas;
- A série possui cobertura de 99,960%. As 91 horas indisponíveis foram mantidas como ausentes, sem preenchimento nesta etapa.

Os resultados também mostraram que a série não é completamente homogênea ao longo do tempo. Além da integração de novos sistemas, o ONS modificou a composição da carga horária em março de 2021 e incorporou a estimativa da micro e minigeração distribuída em abril de 2023. Essas mudanças precisam ser consideradas ao comparar períodos e construir modelos.

### Implicações para as Próximas Etapas

Para previsão e modelagem, as variáveis de calendário identificadas nesta análise — mês, dia da semana, hora, tipo de dia e feriados — podem ajudar a explicar as oscilações recorrentes da carga.

Antes disso, será necessário:

1. definir se o alvo será a carga total do SIN ou um modelo separado para cada subsistema;
2. escolher a frequência de modelagem, como horária ou diária;
3. criar uma sequência temporal regular e decidir como tratar as 91 horas indisponíveis;
4. avaliar os registros mínimos atípicos e o efeito das mudanças de abrangência e metodologia;
5. separar treino, validação e teste respeitando a ordem temporal;
6. construir modelos de referência antes de testar métodos mais avançados.

## 15. Fontes de Contexto

- [ONS — Histórico da Carga de Energia e mudanças metodológicas](https://www.ons.org.br/Paginas/resultados-da-operacao/historico-da-operacao/carga_energia.aspx)
- [ONS — Integração do Amazonas ao SIN em julho de 2013](https://www.ons.org.br/sites/multimidia/Documentos%20Compartilhados/dados/dados_relevantes_2013/HTML/01-07-destaques.html)
- [ONS — Boletim Mensal de Carga de abril de 2020](https://www.ons.org.br/AcervoDigitalDocumentosEPublicacoes/BoletimMensalCarga_abril-2020%20%28002%29.pdf)
- [EPE — PNE 2030: Análise Retrospectiva](https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/publicacoes/PublicacoesArquivos/publicacao-165/topico-173/PNE%202030%20-%20An%C3%A1lise%20Retrospectiva.pdf)
- [Ministério de Minas e Energia — Horário de Verão](https://www.gov.br/mme/pt-br/assuntos/secretarias/secretaria-nacional-energia-eletrica/horario-de-verao)